# 01 · Bronze — Ingesta cruda

| | |
|---|---|
| **Objetivo** | Materializar el archivo de origen como tabla Delta sin alterar ningun valor |
| **Entradas** | `train.csv` o `test.csv` en el volumen de aterrizaje |
| **Salidas** | `bronze.raw_trips` (entrenamiento) o `bronze.raw_scoring` (lote de scoring) |
| **Parametro** | `p_origen` = `train` \| `scoring` |
| **Depende de** | `00_setup` ejecutado y los archivos cargados en el volumen |

**Proceso**
1. Leer el CSV con esquema explicito de tipo texto
2. Anexar columnas tecnicas de auditoria y hash de fila
3. Escribir la tabla Delta
4. Verificar el conteo contra el publicado por la fuente

**Principio de la capa:** Bronze es un espejo fiel del archivo. Todo se lee
como texto para que ningun tipo se interprete prematuramente; lo unico que
se agrega son columnas tecnicas. El tipado y la limpieza corresponden a
Silver.

El mismo notebook ingesta ambos conjuntos, que tienen esquemas distintos:
el de scoring no incluye `dropoff_datetime` ni `trip_duration`.

In [0]:
import os
import sys


# Localiza la raiz del repo subiendo hasta encontrar src/, en vez de fijar un
# numero de saltos. Asi el notebook funciona a cualquier profundidad.
def _preparar_path(marcador="src", max_niveles=10):
    ruta = os.getcwd()
    for _ in range(max_niveles):
        if os.path.isdir(os.path.join(ruta, marcador)):
            destino = os.path.join(ruta, marcador)
            if destino not in sys.path:
                sys.path.insert(0, destino)
            return destino
        padre = os.path.dirname(ruta)
        if padre == ruta:
            break
        ruta = padre
    raise RuntimeError(f"No se encontro la raiz del repo (carpeta con {marcador}/)")


_preparar_path()

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

from nyc_taxi import config

In [0]:
dbutils.widgets.dropdown("p_origen", "train", ["train", "scoring"], "Origen del archivo")

In [0]:
# El set de scoring es el de entrenamiento menos las dos columnas que la fuente
# retira deliberadamente: dropoff_datetime, porque permite derivar el target por
# simple resta, y trip_duration, que es el target. Esa ausencia es la evidencia
# documental de qué información NO está disponible al momento de predecir.
COLS_TRAIN = [
    "id", "vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count",
    "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
    "store_and_fwd_flag", "trip_duration",
]
COLS_SCORING = [c for c in COLS_TRAIN if c not in ("dropoff_datetime", "trip_duration")]

ORIGENES = {
    "train": {
        "archivo": "train.csv",
        "tabla": config.TBL_RAW_TRIPS,
        "columnas": COLS_TRAIN,
        "filas_esperadas": config.FILAS_TRAIN,
    },
    "scoring": {
        "archivo": "test.csv",
        "tabla": config.TBL_RAW_SCORING,
        "columnas": COLS_SCORING,
        "filas_esperadas": config.FILAS_SCORING,
    },
}

origen = dbutils.widgets.get("p_origen")
cfg = ORIGENES[origen]
ruta = f"{config.VOLUMEN_LANDING}/{cfg['archivo']}"

print(f"Origen : {origen}\nArchivo: {ruta}\nDestino: {cfg['tabla']}")

Origen : train
Archivo: /Volumes/nyc_taxi/bronze/landing/train.csv
Destino: nyc_taxi.bronze.raw_trips


## Lectura

Schema explícito de tipo texto en vez de `inferSchema`. Evita una pasada
completa extra sobre los 191 MB solo para adivinar tipos, y garantiza que
Bronze no altere ningún valor.

`FAILFAST` hace que un archivo que no calce con el schema reviente aquí y
no tres pasos después: por defecto Spark pondría nulos y seguiría.

In [0]:
schema = StructType([StructField(c, StringType(), True) for c in cfg["columnas"]])

df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(schema)
    .load(ruta)
)

## Columnas de auditoría

El hash SHA-256 de la fila detecta duplicados exactos entre cargas y da
idempotencia a los reprocesos. Se calcula solo sobre columnas de negocio
—nunca sobre las de auditoría— para que reingestar el mismo archivo
produzca el mismo hash.

Los nulos se sustituyen por un centinela porque `concat_ws` los omite, y
sin eso dos filas distintas podrían colisionar en el mismo hash.

In [0]:
cols_hash = [F.coalesce(F.col(c), F.lit("<null>")) for c in cfg["columnas"]]

df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(cfg["archivo"]))
    .withColumn("_origen", F.lit(origen))
    .withColumn("_row_hash", F.sha2(F.concat_ws("||", *cols_hash), 256))
)

In [0]:
(
    df_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg["tabla"])
)

## Verificación de carga

Comparar contra el conteo publicado por la fuente detecta de inmediato una
subida truncada o un archivo mal delimitado.

Los dos conteos de duplicados detectan cosas distintas: un `id` repetido
con hash distinto es un reenvío corregido; un hash repetido es duplicación
real del mismo registro.

In [0]:
df_check = spark.table(cfg["tabla"])
n_filas = df_check.count()
n_hash = df_check.select("_row_hash").distinct().count()
n_ids = df_check.select("id").distinct().count()

print(f"Filas cargadas   : {n_filas:,}")
print(f"Filas esperadas  : {cfg['filas_esperadas']:,}")
print(f"Hashes distintos : {n_hash:,}   (duplicados exactos: {n_filas - n_hash:,})")
print(f"IDs distintos    : {n_ids:,}   (ids repetidos: {n_filas - n_ids:,})")

assert n_filas == cfg["filas_esperadas"], (
    f"Conteo inesperado: {n_filas:,} vs {cfg['filas_esperadas']:,}. "
    "Revisar si la subida del archivo quedó incompleta."
)

Filas cargadas   : 1,458,644
Filas esperadas  : 1,458,644
Hashes distintos : 1,458,644   (duplicados exactos: 0)
IDs distintos    : 1,458,644   (ids repetidos: 0)


In [0]:
display(df_check.limit(10))